# ow-0009 (April 2019) -- candidate regen + AIS-3/4/5 (matching, evidence, ranking)

**Same unmodified `iste.*` modules as the June event's AIS-3/4/5 -- new event's
own candidates/vessels only.** T14-T17 candidate sweep + T36 (ais_source=None,
for cluster-id lookup only) + T37 clustering, then AIS-3 (spatial-temporal
matching), AIS-4 (evidence aggregation), AIS-5 (vessel ranking) -- each module
called exactly as already approved, consolidated into one notebook purely to
avoid re-deriving the same candidate sweep four separate times.

In [1]:
import os, sys, time, json as _json
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from iste.ensemble_run import EnsembleRunRecords
from iste.analysis_grid import generate_analysis_grid
from iste.candidate_extraction import CandidateExtractionConfig, find_candidate_peaks
from iste.kdtree_neighborhood import build_particle_neighborhood_index
from iste.particle_query import query_particle_slice
from iste.support_features import SupportFeatureConfig, compute_support_map
from iste.phase3_composite import DEFAULT_WEIGHTS, compute_source_scores_for_candidates
from iste.phase3_clustering import DEFAULT_EPS_M, cluster_candidates
from iste.ais_matching import (
    haversine_distance_km, analyze_timestamp_alignment, compute_vessel_candidate_matches,
    summarize_vessel_matches, analyze_vessel_track, DEFAULT_DISTANCE_BINS_KM,
    DEFAULT_TIME_TOLERANCE_HOURS, DEFAULT_CLOSE_ENCOUNTER_DISTANCE_KM,
)
from iste.ais_evidence import compute_vessel_evidence, compute_checkpoint_proximity, run_circular_shift_null
from iste.ais_ranking import (
    AISScoreWeights, DEFAULT_WEIGHTS as AIS5_DEFAULT_WEIGHTS, DEFAULT_SPATIAL_SCALE_KM,
    compute_ais_ranking, compute_evidence_concentration, EVIDENCE_RANKING_DISCLAIMER,
)

RESULT_DIR = os.path.join(REPO_ROOT, "result_ow0009")
os.makedirs(RESULT_DIR, exist_ok=True)

## 1. Load Stage III particle history (just computed for ow-0009)

In [2]:
history_path = os.path.join(RESULT_DIR, "stage3_particle_history_ow0009.npz")
cached = np.load(history_path, allow_pickle=True)
def _none_if_empty(arr):
    return None if arr.size == 0 else arr
records = EnsembleRunRecords(
    particle_id=cached["particle_id"], forcing_member_id=cached["forcing_member_id"],
    observation_pixel_id=cached["observation_pixel_id"], timestamp=cached["timestamp"],
    lat=cached["lat"], lon=cached["lon"], log_weight=cached["log_weight"], valid=cached["valid"],
    death_time=_none_if_empty(cached["death_time"]), death_lat=_none_if_empty(cached["death_lat"]),
    death_lon=_none_if_empty(cached["death_lon"]), death_reason=_none_if_empty(cached["death_reason"]),
    survival_mark=_none_if_empty(cached["survival_mark"]),
)
T_OBS = np.datetime64(np.max(records.timestamp), "ns")
grid = generate_analysis_grid(T_OBS, window_hours=72.0, interval_hours=1.0)
historical_timestamps = grid[1:]
print(f"T0={T_OBS}, {len(grid)} checkpoints, {len(records.timestamp)} particle-history rows")

T0=2019-04-28T15:38:41.000000000, 73 checkpoints, 162498 particle-history rows


## 2. T14-T17 candidate sweep (identical, unmodified)

In [3]:
spill_data = _json.load(open(os.path.join(REPO_ROOT, "data", "database_payload.json"), encoding="utf-8"))
spill = next(s for s in spill_data["spills"] if s["image_filename"] == "ow-0009.jpg")
obs_bbox = spill["bounding_box"]
valid_mask = records.valid
lat_min = min(obs_bbox["upper_left"]["lat"], obs_bbox["bottom_right"]["lat"], float(records.lat[valid_mask].min()))
lat_max = max(obs_bbox["upper_left"]["lat"], obs_bbox["bottom_right"]["lat"], float(records.lat[valid_mask].max()))
lon_min = min(obs_bbox["upper_left"]["lon"], obs_bbox["bottom_right"]["lon"], float(records.lon[valid_mask].min()))
lon_max = max(obs_bbox["upper_left"]["lon"], obs_bbox["bottom_right"]["lon"], float(records.lon[valid_mask].max()))
PAD_DEG = 0.03
support_lat_grid = np.arange(lat_min - PAD_DEG, lat_max + PAD_DEG, 0.003)
support_lon_grid = np.arange(lon_min - PAD_DEG, lon_max + PAD_DEG, 0.003)
NEIGHBORHOOD_RADIUS_M = 1000.0
support_config = SupportFeatureConfig()
extraction_config = CandidateExtractionConfig(peak_neighborhood_cells=5, min_separation_m=1500.0, core_radius_m=1000.0)
lon_mesh, lat_mesh = np.meshgrid(support_lon_grid, support_lat_grid)

t0 = time.time()
all_candidates = []
support_grids_by_t_s = {}
for t_s in historical_timestamps:
    slice_ = query_particle_slice(records, t_s)
    index = build_particle_neighborhood_index(slice_)
    support_map = compute_support_map(index, lat_mesh.ravel(), lon_mesh.ravel(), radius_m=NEIGHBORHOOD_RADIUS_M, config=support_config)
    support_grids_by_t_s[np.datetime64(t_s, "ns")] = np.array([sf.support for sf in support_map]).reshape(len(support_lat_grid), len(support_lon_grid))
    all_candidates.extend(find_candidate_peaks(support_map, support_lat_grid, support_lon_grid, t_s, config=extraction_config))
t1 = time.time()
print(f"regenerated {len(all_candidates)} candidates over {len(historical_timestamps)} historical timestamps in {t1-t0:.1f}s")

hotspots_export = [
    {"candidate_id": c.candidate_id, "lat": c.y_peak, "lon": c.x_peak,
     "t_s": str(np.datetime_as_string(np.datetime64(c.t_s, "ns"), unit="s")),
     "support_score": c.support_score, "density_score": c.density_score,
     "weighted_support_score": c.weighted_support_score, "pixel_diversity_score": c.pixel_diversity_score}
    for c in all_candidates
]
with open(os.path.join(RESULT_DIR, "hotspots.json"), "w", encoding="utf-8") as f:
    _json.dump(hotspots_export, f, indent=2)
print("saved hotspots.json")

regenerated 1647 candidates over 72 historical timestamps in 216.9s
saved hotspots.json


## 3. T36 (ais_source=None) + T37 clustering -- for cluster_id lookup only, same as June's AIS-3 pattern

In [4]:
scores_no_ais = compute_source_scores_for_candidates(
    records, all_candidates, support_grids_by_t_s, neighborhood_radius_m=NEIGHBORHOOD_RADIUS_M, weights=DEFAULT_WEIGHTS,
)
clusters_no_ais = cluster_candidates(all_candidates, scores_no_ais, eps_m=DEFAULT_EPS_M)
print(f"T36 (no AIS) + T37: {len(clusters_no_ais)} clusters")

cluster_lookup = {}
for cl in clusters_no_ais:
    for cid in cl.member_candidate_ids:
        cluster_lookup[cid] = cl.cluster_id

candidates_df = pd.DataFrame([
    dict(candidate_id=c.candidate_id, t_s=pd.Timestamp(np.datetime64(c.t_s, "ns"), tz="UTC"),
         lat=c.y_peak, lon=c.x_peak, support_score=c.support_score, cluster_id=cluster_lookup[c.candidate_id])
    for c in all_candidates
])
print(f"candidates_df: {len(candidates_df)} rows, {candidates_df['t_s'].nunique()} distinct timestamps")

ais_path = os.path.join(RESULT_DIR, "AIS_vessel_tracks_event_window.csv")
ais_df = pd.read_csv(ais_path)
ais_df["timestamp"] = pd.to_datetime(ais_df["timestamp"], utc=True)
print(f"loaded {len(ais_df)} AIS observations ({ais_df['mmsi'].nunique()} vessels)")

T36 (no AIS) + T37: 7 clusters
candidates_df: 1647 rows, 72 distinct timestamps
loaded 8017 AIS observations (217 vessels)


## 4. AIS-3: spatial-temporal matching (same module, unmodified)

In [5]:
alignment = analyze_timestamp_alignment(candidates_df["t_s"], ais_df["timestamp"])
print(f"n_candidate_timestamps={alignment.n_candidate_timestamps}, n_distinct_ais_timestamps={alignment.n_distinct_ais_timestamps}, "
      f"fraction_ais_on_grid={alignment.fraction_ais_timestamps_on_grid:.3f}")

t0 = time.time()
matches_df = compute_vessel_candidate_matches(candidates_df, ais_df, max_time_tolerance_hours=DEFAULT_TIME_TOLERANCE_HOURS)
t1 = time.time()
print(f"computed {len(matches_df)} (candidate, AIS observation) pairs within +/-{DEFAULT_TIME_TOLERANCE_HOURS}h in {t1-t0:.1f}s")
print(f"is_exact: {int(matches_df['is_exact'].sum())}, is_within_1h: {int(matches_df['is_within_1h'].sum())}, is_within_2h: {int(matches_df['is_within_2h'].sum())}")

vessel_summary_df = summarize_vessel_matches(
    matches_df, ais_df, n_candidate_timestamps=candidates_df["t_s"].nunique(), distance_bins_km=DEFAULT_DISTANCE_BINS_KM,
)
matches_df.to_csv(os.path.join(RESULT_DIR, "AIS3_vessel_candidate_matches.csv"), index=False)
vessel_summary_df.to_csv(os.path.join(RESULT_DIR, "AIS3_vessel_match_summary.csv"), index=False)
print("saved AIS3_vessel_candidate_matches.csv, AIS3_vessel_match_summary.csv")

n_candidate_timestamps=72, n_distinct_ais_timestamps=73, fraction_ais_on_grid=0.986


computed 891632 (candidate, AIS observation) pairs within +/-2.0h in 80.3s
is_exact: 183147, is_within_1h: 541560, is_within_2h: 891632


saved AIS3_vessel_candidate_matches.csv, AIS3_vessel_match_summary.csv


## 5. AIS-4: aggregate vessel evidence (same module, unmodified)

In [6]:
gap_summary_df = pd.read_csv(os.path.join(RESULT_DIR, "AIS_temporal_gap_summary.csv")) if os.path.exists(os.path.join(RESULT_DIR, "AIS_temporal_gap_summary.csv")) else None
if gap_summary_df is None:
    from iste.ais_data import build_vessel_tracks, compute_track_temporal_gaps
    _tracks = build_vessel_tracks(ais_df.rename(columns={"timestamp": "timestamp"}))
    gap_summary_df = pd.DataFrame([
        dict(mmsi=t.mmsi, vessel_name=t.vessel_name, vessel_type=t.vessel_type,
             n_observations=g.n_observations,
             median_interval_hours=(round(g.median_interval_hours, 3) if not np.isnan(g.median_interval_hours) else None),
             max_interval_hours=(round(g.max_interval_hours, 3) if not np.isnan(g.max_interval_hours) else None),
             n_gaps_over_threshold=g.n_gaps_over_threshold)
        for t in _tracks for g in [compute_track_temporal_gaps(t, gap_threshold_hours=6.0)]
    ])
    gap_summary_df.to_csv(os.path.join(RESULT_DIR, "AIS_temporal_gap_summary.csv"), index=False)

n_candidate_timestamps = int(candidates_df["t_s"].nunique())
candidate_t_s_grid = sorted(candidates_df["t_s"].unique())

t0 = time.time()
vessel_evidence_df = compute_vessel_evidence(
    matches_df, ais_df, gap_summary_df, n_candidate_timestamps,
    distance_bins_km=DEFAULT_DISTANCE_BINS_KM, close_encounter_km=DEFAULT_CLOSE_ENCOUNTER_DISTANCE_KM,
)
t1 = time.time()
print(f"computed vessel evidence for {len(vessel_evidence_df)} vessels in {t1-t0:.1f}s")
vessel_evidence_df.to_csv(os.path.join(RESULT_DIR, "AIS4_vessel_evidence.csv"), index=False)

checkpoint_proximity_df = compute_checkpoint_proximity(candidates_df, matches_df)
checkpoint_proximity_df.to_csv(os.path.join(RESULT_DIR, "AIS4_checkpoint_proximity.csv"), index=False)
print("saved AIS4_vessel_evidence.csv, AIS4_checkpoint_proximity.csv")

computed vessel evidence for 217 vessels in 5.6s


saved AIS4_vessel_evidence.csv, AIS4_checkpoint_proximity.csv


In [7]:
t0 = time.time()
null_rows = []
for mmsi in ais_df["mmsi"].unique():
    result = run_circular_shift_null(int(mmsi), candidates_df, ais_df, candidate_t_s_grid)
    if result is None:
        continue
    null_rows.append(dict(
        mmsi=result.mmsi, real_min_distance_km=result.real_min_distance_km,
        n_shifts_tested=result.n_shifts_tested, null_median_km=result.null_median_km,
        null_p10_km=result.null_p10_km, empirical_p_value=result.empirical_p_value,
    ))
t1 = time.time()
null_df = pd.DataFrame(null_rows).sort_values("empirical_p_value") if null_rows else pd.DataFrame(
    columns=["mmsi","real_min_distance_km","n_shifts_tested","null_median_km","null_p10_km","empirical_p_value"])
print(f"ran circular-shift null for {len(null_df)} / {ais_df['mmsi'].nunique()} vessels in {t1-t0:.1f}s")
null_df.to_csv(os.path.join(RESULT_DIR, "AIS4_null_diagnostics.csv"), index=False)
print("saved AIS4_null_diagnostics.csv")

ran circular-shift null for 212 / 217 vessels in 36.5s
saved AIS4_null_diagnostics.csv


## 6. AIS-5: vessel ranking (same module, unmodified)

In [8]:
ranking_primary = compute_ais_ranking(vessel_evidence_df, matches_df, null_df, candidates_df, level="exact", weights=AIS5_DEFAULT_WEIGHTS)
print(f"ranked {len(ranking_primary)} vessels; {int(ranking_primary['ais_attribution_score'].notna().sum())} with a defined score")
ranking_primary.to_csv(os.path.join(RESULT_DIR, "AIS5_vessel_ranking.csv"), index=False)
print("saved AIS5_vessel_ranking.csv")
print(ranking_primary[["rank","mmsi","vessel_name","ais_attribution_score","strongest_encounter_distance_km","strongest_candidate_cluster_id"]].head(15).to_string(index=False))

ranked 217 vessels; 217 with a defined score
saved AIS5_vessel_ranking.csv
 rank      mmsi           vessel_name  ais_attribution_score  strongest_encounter_distance_km strongest_candidate_cluster_id
  1.0 211482965   SEASPAN CARRIER U94               0.369634                         0.107833                    cluster0000
  2.0 370978298         MT SPIRIT Y80               0.329678                         2.094724                    cluster0000
  3.0 352510260       OCEAN GLORY U47               0.308764                        11.751770                    cluster0000
  4.0 256383837     NORDIC SPIRIT W40               0.260247                         3.425649                    cluster0000
  5.0 352844541     PACIFIC JEWEL Z77               0.229153                         5.958572                    cluster0000
  6.0 352291216          MV EAGLE B40               0.220648                        15.111847                    cluster0000
  7.0 538570865 MINERVA NAVIGATOR J50             

## 7. Validation

In [9]:
checks = []
checks.append(("AIS3/4/5 output files exist", all(os.path.exists(os.path.join(RESULT_DIR, f)) for f in
    ["AIS3_vessel_candidate_matches.csv", "AIS4_vessel_evidence.csv", "AIS5_vessel_ranking.csv"])))
checks.append(("every vessel in ais_df appears in vessel_evidence_df", set(ais_df["mmsi"]) == set(vessel_evidence_df["mmsi"])))
checks.append(("ranking reproducible", compute_ais_ranking(vessel_evidence_df, matches_df, null_df, candidates_df, level="exact", weights=AIS5_DEFAULT_WEIGHTS)["ais_attribution_score"].equals(ranking_primary["ais_attribution_score"])))
checks.append(("null p-values in [0,1]", bool(((null_df["empirical_p_value"] >= 0) & (null_df["empirical_p_value"] <= 1)).all()) if len(null_df) else True))
for label, ok in checks:
    print(f"[{'OK' if ok else 'FAIL'}] {label}")
assert all(ok for _, ok in checks)

output_files = sorted(f for f in os.listdir(RESULT_DIR) if f.startswith(("AIS3_", "AIS4_", "AIS5_", "hotspots")))
print(f"\n{len(output_files)} files saved under {RESULT_DIR}:")
for fn in output_files:
    print(f"  {fn}")

[OK] AIS3/4/5 output files exist
[OK] every vessel in ais_df appears in vessel_evidence_df
[OK] ranking reproducible
[OK] null p-values in [0,1]

7 files saved under C:\Users\venka\OneDrive\Desktop\SIH-2026\project-v5\result_ow0009:
  AIS3_vessel_candidate_matches.csv
  AIS3_vessel_match_summary.csv
  AIS4_checkpoint_proximity.csv
  AIS4_null_diagnostics.csv
  AIS4_vessel_evidence.csv
  AIS5_vessel_ranking.csv
  hotspots.json
